# Neo4j Python Graph Visualization
* https://neo4j.com/docs/python-graph-visualization/current/

# Setup

In [10]:
!pip install neo4j-viz
!pip install neo4j-viz[neo4j]
!pip install neo4j-viz[gds]
!pip install neo4j-viz[pandas]

In [12]:
!pip install python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymilvus 2.5.14 requires grpcio<=1.67.1,>=1.49.1, but you have grpcio 1.78.0 which is incompatible.


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

# Getting started

In [17]:
from neo4j_viz import Node, Relationship, VisualizationGraph

nodes = [
    Node(id=0, size=10, caption="Person"),
    Node(id=1, size=10, caption="Product"), # node size
    Node(id=2, size=20, caption="Product"),
    Node(id=3, size=10, caption="Person"),
    Node(id=4, size=10, caption="Product"),
]

relationships = [
    Relationship(
        source=0,
        target=1,
        caption="BUYS",
    ),
    Relationship(
        source=0,
        target=2,
        caption="BUYS",
    ),
    Relationship(
        source=3,
        target=2,
        caption="BUYS",
    ),
]

VG = VisualizationGraph(nodes=nodes, relationships=relationships)
# VG.color_nodes(field='caption')

# VG.render(initial_zoom=2)

# Rendering a graph

```python
render(layout=None, 
  layout_options=None, 
  renderer=Renderer.CANVAS, 
  width='100%', 
  height='600px', 
  pan_position=None, 
  initial_zoom=None, 
  min_zoom=0.075, 
  max_zoom=10, 
  allow_dynamic_min_zoom=True, 
  max_allowed_nodes=10000, 
  theme='auto')
```

In [ ]:
# Exporting to HTML

html = VG.render()
with open("my_graph.html", "w") as f:
  f.write(html.data)

# Customizing the visualization
- Setting node captions: `VG.set_node_captions()`
- Coloring the graph: `VG.color_nodes()`, `VG.color_relationships()`
- Sizing nodes and relationships: `VG.resize_nodes()`, `VG.resize_relationships()`
- Pinning nodes: `VG.toggle_nodes_pinned()`
- Direct modification of nodes and relationships: Nodes and relationships can also be modified directly by accessing the `nodes` and `relationships` fields of an existing `VisualizationGraph` object.

# Visualizing existing data

## Neo4j database

In [14]:
from neo4j import GraphDatabase, RoutingControl, Result
from neo4j_viz.neo4j import from_neo4j
import os

# Modify this to match your Neo4j instance's URI and credentials
URI = "neo4j://localhost:7687"
auth = ("neo4j", os.getenv('NEO4J_AUTH', default='neo4j'))

with GraphDatabase.driver(URI, auth=auth) as driver:
    driver.verify_connectivity()
    result = driver.execute_query(
        "MATCH (n)-[r]->(m) RETURN n,r,m",
        database_="neo4j",
        routing_=RoutingControl.READ,
        result_transformer_=Result.graph,
    )

VG = from_neo4j(result)
# VG.render()

## Neo4j Graph Data Science

In [5]:
from graphdatascience import GraphDataScience
import os

URI = "neo4j://localhost:7687"
auth = ("neo4j", os.getenv('NEO4J_AUTH', default='neo4j'))
db = os.getenv('NEO4J_DB', 'neo4j')

gds = GraphDataScience(URI, auth=auth, database=db)

In [ ]:
G = gds.graph.load_cora(graph_name="cora")

gds.nodeSimilarity.mutate(
    G, mutateRelationshipType="SIMILAR", mutateProperty="similarity"
)
gds.pageRank.mutate(G, mutateProperty="pagerank")
gds.louvain.mutate(G, mutateProperty="componentId")

 Louvain:  10%|9         | 9.99/100 [00:00<?, ?%/s]

mutateMillis                                                             1
nodePropertiesWritten                                                 2708
modularity                                                        0.722776
modularities             [0.46256109609549334, 0.6231609064132905, 0.67...
ranLevels                                                               10
communityCount                                                         534
communityDistribution    {'p1': 1, 'max': 184, 'p5': 1, 'p90': 2, 'p50'...
postProcessingMillis                                                     6
preProcessingMillis                                                      0
computeMillis                                                         3437
configuration            {'maxIterations': 10, 'seedProperty': None, 'c...
Name: 0, dtype: object

In [13]:
from neo4j_viz.gds import from_gds

VG = from_gds(gds, G, 
              # node_properties=["componentId"],
              )
# VG.color_nodes(property="componentId")
# VG.render(theme="auto")
# too big!!!

## GQL CREATE queries

In [15]:
from neo4j_viz.gql_create import from_gql_create

query = """
        CREATE
          (a:User {name: 'Alice', age: 23}),
          (b:User {name: 'Bridget', age: 34}),
          (c:User {name: 'Charles', age: 45}),
          (d:User {name: 'Dana', age: 56}),
          (e:User {name: 'Eve', age: 67}),
          (f:User {name: 'Fawad', age: 78}),
          (a)-[:LINK {weight: 0.5}]->(b),
          (a)-[:LINK {weight: 4}]->(c),
          (e)-[:LINK {weight: 1.1}]->(d),
          (e)-[:LINK {weight: -2}]->(f);
        """

VG = from_gql_create(query)
# VG.render(initial_zoom=2)

## Pandas DataFrames

In [16]:
from pandas import DataFrame
from neo4j_viz.pandas import from_dfs

nodes = DataFrame({
    "id": [1, 2, 3],
    "caption": ["Alice", "Bob", "Charlie"],
    "size": [20, 10, 10],
})

relationships = DataFrame({
    "source": [1, 2],
    "target": [2, 3],
    "caption": ["LIKES", "KNOWS"],
})

VG = from_dfs(nodes, relationships)
# VG.render(initial_zoom=2)

# Cleanup

In [8]:
import os

to_del_files = ['my_graph.html']
for to_del_file in to_del_files:
  if os.path.exists(to_del_file):
    os.remove(to_del_file)